In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 60)

df = pd.read_csv("../data/disprot.tsv", sep='\t', low_memory=False)
print(df.shape, df.columns.tolist())

(13396, 21) ['UniProt ACC', 'DisProt ID', 'Protein name', 'Gene name', 'Sequence length', 'Organism', 'NCBI Taxon ID', 'Protein Disorder Content', 'Region ID', 'Start', 'End', 'Term namespace', 'Term ID', 'Term name', 'ECO Term ID', 'ECO Term name', 'PMID', 'Region sequence', 'Alphafold Very Low confidence content', 'Obsolete', 'Dataset']


In [3]:
df = df.rename(columns={
    'UniProt ACC': 'acc',
    'DisProt ID': 'disprot_id',
    'Protein name': 'name',
    'Gene name': 'gene_name',
    'Sequence length': 'length',
    'Organism': 'organism',
    'NCBI Taxon ID': 'ncbi_taxon_id',
    'Protein Disorder Content': 'disorder_content',
    'Region ID': 'region_id',
    'Start': 'start',
    'End': 'end',
    'Term namespace': 'term_namespace',
    'Term ID': 'term',
    'Term name': 'term_name',
    'ECO Term ID': 'ec',
    'ECO Term name': 'ec_name',
    'PMID': 'pmid',
    'Region sequence': 'region_sequence',
    'Alphafold Very Low confidence content': 'alphafold_lc',
    'Obsolete': 'obsolete',
    'Dataset': 'dataset',
})
print(df.columns.tolist())

['acc', 'disprot_id', 'name', 'gene_name', 'length', 'organism', 'ncbi_taxon_id', 'disorder_content', 'region_id', 'start', 'end', 'term_namespace', 'term', 'term_name', 'ec', 'ec_name', 'pmid', 'region_sequence', 'alphafold_lc', 'obsolete', 'dataset']


In [4]:
human = df[df['organism'] == 'Homo sapiens']
candidates = sorted(human['acc'].unique())
print(f"Candidate human proteins: {len(candidates)}")

pd.Series(candidates, name='acc').to_csv("../data/candidate_accs.csv", index=False)

Candidate human proteins: 1301


In [5]:
import requests
from io import StringIO
from Bio import SeqIO

accs = pd.read_csv("../data/candidate_accs.csv")['acc'].tolist()
batch = 500
records = []
for i in range(0, len(accs), batch):
    chunk = accs[i:i+batch]
    query = "+OR+".join(f"accession:{a}" for a in chunk)
    url = f"https://rest.uniprot.org/uniprotkb/stream?query={query}&format=fasta"
    r = requests.get(url, timeout=180); r.raise_for_status()
    records.extend(SeqIO.parse(StringIO(r.text), 'fasta'))
print(f"Retrieved {len(records)} sequences")

SeqIO.write(records, "../data/sequences.fasta", "fasta")
seq_df = pd.DataFrame([{
    'acc': rec.id.split('|')[1] if '|' in rec.id else rec.id,
    'sequence': str(rec.seq),
    'length': len(rec.seq),
} for rec in records])
seq_df.to_csv("../data/sequences.csv", index=False)
print(seq_df['length'].describe())

HTTPError: 400 Client Error:  for url: https://rest.uniprot.org/uniprotkb/stream?query=accession:A0A090N8E9+OR+accession:A0A0C4DGY3+OR+accession:A0A1B0GTR3+OR+accession:A0A5F9ZHA1+OR+accession:A4D126+OR+accession:A5YKK6+OR+accession:A6ND01+OR+accession:A6NF83+OR+accession:A6NI73+OR+accession:A8K2U0+OR+accession:A8MTZ0+OR+accession:O00189+OR+accession:O00204+OR+accession:O00204-2+OR+accession:O00214+OR+accession:O00268+OR+accession:O00273+OR+accession:O00299+OR+accession:O00308+OR+accession:O00410+OR+accession:O00411+OR+accession:O00418+OR+accession:O00429-3+OR+accession:O00488+OR+accession:O00499+OR+accession:O00522+OR+accession:O00548+OR+accession:O00571+OR+accession:O00584+OR+accession:O00585+OR+accession:O14497+OR+accession:O14519+OR+accession:O14558+OR+accession:O14672+OR+accession:O14678+OR+accession:O14686+OR+accession:O14713+OR+accession:O14727+OR+accession:O14733+OR+accession:O14745+OR+accession:O14746+OR+accession:O14775+OR+accession:O14776+OR+accession:O14810+OR+accession:O14

In [6]:
# Strip isoform suffixes (e.g., O00204-2 -> O00204) so accessions match what UniProt expects
df['acc'] = df['acc'].str.split('-').str[0]

# Re-derive the candidate list and overwrite the CSV
human = df[df['organism'] == 'Homo sapiens']
candidates = sorted(human['acc'].unique())
print(f"Base accessions after deduping isoforms: {len(candidates)}")
pd.Series(candidates, name='acc').to_csv("../data/candidate_accs.csv", index=False)

Base accessions after deduping isoforms: 1279


In [7]:
import requests
from io import StringIO
from Bio import SeqIO

accs = pd.read_csv("../data/candidate_accs.csv")['acc'].tolist()
batch = 100
records = []
n_batches = (len(accs) + batch - 1) // batch
for i in range(0, len(accs), batch):
    chunk = accs[i:i+batch]
    query = " OR ".join(f"accession:{a}" for a in chunk)
    r = requests.get(
        "https://rest.uniprot.org/uniprotkb/stream",
        params={"query": query, "format": "fasta"},
        timeout=180,
    )
    r.raise_for_status()
    records.extend(SeqIO.parse(StringIO(r.text), 'fasta'))
    print(f"  Batch {i//batch + 1}/{n_batches}: {len(records)} cumulative")
print(f"\nRetrieved {len(records)} sequences")

SeqIO.write(records, "../data/sequences.fasta", "fasta")
seq_df = pd.DataFrame([{
    'acc': rec.id.split('|')[1] if '|' in rec.id else rec.id,
    'sequence': str(rec.seq),
    'length': len(rec.seq),
} for rec in records])
seq_df.to_csv("../data/sequences.csv", index=False)
print(seq_df['length'].describe())

  Batch 1/13: 100 cumulative
  Batch 2/13: 200 cumulative
  Batch 3/13: 300 cumulative
  Batch 4/13: 400 cumulative
  Batch 5/13: 500 cumulative
  Batch 6/13: 600 cumulative
  Batch 7/13: 700 cumulative
  Batch 8/13: 800 cumulative
  Batch 9/13: 900 cumulative
  Batch 10/13: 1000 cumulative
  Batch 11/13: 1100 cumulative
  Batch 12/13: 1200 cumulative
  Batch 13/13: 1279 cumulative

Retrieved 1279 sequences
count     1279.000000
mean       644.176701
std       1085.087132
min         24.000000
25%        277.000000
50%        482.000000
75%        764.500000
max      34350.000000
Name: length, dtype: float64


In [7]:
import requests
from io import StringIO
from Bio import SeqIO

accs = pd.read_csv("../data/candidate_accs.csv")['acc'].tolist()
batch = 100
records = []
n_batches = (len(accs) + batch - 1) // batch
for i in range(0, len(accs), batch):
    chunk = accs[i:i+batch]
    query = " OR ".join(f"accession:{a}" for a in chunk)
    r = requests.get(
        "https://rest.uniprot.org/uniprotkb/stream",
        params={"query": query, "format": "fasta"},
        timeout=180,
    )
    r.raise_for_status()
    records.extend(SeqIO.parse(StringIO(r.text), 'fasta'))
    print(f"  Batch {i//batch + 1}/{n_batches}: {len(records)} cumulative")
print(f"\nRetrieved {len(records)} sequences")

SeqIO.write(records, "../data/sequences.fasta", "fasta")
seq_df = pd.DataFrame([{
    'acc': rec.id.split('|')[1] if '|' in rec.id else rec.id,
    'sequence': str(rec.seq),
    'length': len(rec.seq),
} for rec in records])
seq_df.to_csv("../data/sequences.csv", index=False)
print(seq_df['length'].describe())

  Batch 1/13: 100 cumulative
  Batch 2/13: 200 cumulative
  Batch 3/13: 300 cumulative
  Batch 4/13: 400 cumulative
  Batch 5/13: 500 cumulative
  Batch 6/13: 600 cumulative
  Batch 7/13: 700 cumulative
  Batch 8/13: 800 cumulative
  Batch 9/13: 900 cumulative
  Batch 10/13: 1000 cumulative
  Batch 11/13: 1100 cumulative
  Batch 12/13: 1200 cumulative
  Batch 13/13: 1279 cumulative

Retrieved 1279 sequences
count     1279.000000
mean       644.176701
std       1085.087132
min         24.000000
25%        277.000000
50%        482.000000
75%        764.500000
max      34350.000000
Name: length, dtype: float64


In [8]:
import urllib.request

url = "http://current.geneontology.org/annotations/goa_human.gaf.gz"
urllib.request.urlretrieve(url, "../data/goa_human.gaf.gz")

cols = ['DB','DB_Object_ID','DB_Object_Symbol','Qualifier','GO_ID','DB_Reference',
        'Evidence_Code','With_From','Aspect','DB_Object_Name','DB_Object_Synonym',
        'DB_Object_Type','Taxon','Date','Assigned_By','Annotation_Extension',
        'Gene_Product_Form_ID']
go = pd.read_csv("../data/goa_human.gaf.gz", sep='\t', comment='!', names=cols,
                 compression='gzip', low_memory=False)
print(f"GOA human total annotations: {len(go)}")

go_candidates = go[go['DB_Object_ID'].isin(accs)].copy()
print(f"Annotations for our candidate set: {len(go_candidates)}")
go_candidates.to_csv("../data/go_annotations_raw.csv", index=False)

print("\nAspect distribution:")
print(go_candidates['Aspect'].value_counts())   # P=BP, F=MF, C=CC
print("\nProteins with at least one GO annotation:",
      go_candidates['DB_Object_ID'].nunique())

HTTPError: HTTP Error 403: Forbidden

In [9]:
import requests

urls = [
    "https://current.geneontology.org/annotations/goa_human.gaf.gz",
    "https://ftp.ebi.ac.uk/pub/databases/GO/goa/HUMAN/goa_human.gaf.gz",
]
for url in urls:
    try:
        print(f"Trying {url} ...")
        r = requests.get(url, timeout=180,
                         headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        with open("../data/goa_human.gaf.gz", "wb") as f:
            f.write(r.content)
        print(f"  Downloaded {len(r.content)/1e6:.1f} MB")
        break
    except Exception as e:
        print(f"  Failed: {type(e).__name__}: {e}")
else:
    raise RuntimeError("All GOA download URLs failed")

cols = ['DB','DB_Object_ID','DB_Object_Symbol','Qualifier','GO_ID','DB_Reference',
        'Evidence_Code','With_From','Aspect','DB_Object_Name','DB_Object_Synonym',
        'DB_Object_Type','Taxon','Date','Assigned_By','Annotation_Extension',
        'Gene_Product_Form_ID']
go = pd.read_csv("../data/goa_human.gaf.gz", sep='\t', comment='!', names=cols,
                 compression='gzip', low_memory=False)
print(f"\nGOA human total annotations: {len(go)}")

go_candidates = go[go['DB_Object_ID'].isin(accs)].copy()
print(f"Annotations for our candidate set: {len(go_candidates)}")
go_candidates.to_csv("../data/go_annotations_raw.csv", index=False)

print("\nAspect distribution:")
print(go_candidates['Aspect'].value_counts())
print("\nProteins with at least one GO annotation:",
      go_candidates['DB_Object_ID'].nunique())

Trying https://current.geneontology.org/annotations/goa_human.gaf.gz ...
  Downloaded 15.0 MB

GOA human total annotations: 906445
Annotations for our candidate set: 129684

Aspect distribution:
Aspect
F    63092
C    38324
P    28268
Name: count, dtype: int64

Proteins with at least one GO annotation: 1271


In [10]:
d2o = df[(df['organism']=='Homo sapiens') & (df['term']=='IDPO:0000011')]
positive_accs = set(d2o['acc'].unique())
print(f"Positive (d2o) proteins: {len(positive_accs)}")

labels = pd.DataFrame({'acc': candidates})
labels['d2o'] = labels['acc'].isin(positive_accs).astype(int)
labels.to_csv("../data/labels.csv", index=False)
print(labels['d2o'].value_counts())

Positive (d2o) proteins: 188
d2o
0    1091
1     188
Name: count, dtype: int64


In [11]:
master = (labels
          .merge(seq_df, on='acc', how='left')
          .merge(go_candidates.groupby('DB_Object_ID').size()
                              .rename('n_go_raw').reset_index()
                              .rename(columns={'DB_Object_ID':'acc'}),
                 on='acc', how='left'))
master['n_go_raw'] = master['n_go_raw'].fillna(0).astype(int)
master.to_csv("../data/master_raw.csv", index=False)

print(master.head())
print(f"\nMissing sequence: {master['sequence'].isna().sum()}")
print(f"Zero GO annotations (raw): {(master['n_go_raw']==0).sum()}")

          acc  d2o                                           sequence  length  \
0  A0A090N8E9    0  MGQTGKKSEKGPVCWRKRVKSEYMRLRQLKRFRRADEVKSMFSSNR...     751   
1  A0A0C4DGY3    0  MSSEMEPLLLAWSYFRRRKFQLCADLCTQMLEKSPYDQAAWILKAR...     505   
2  A0A1B0GTR3    0  MAKVTSEPQKPNEDVDEQTPSTSSTKGRKKGKTPRQRRSRSGVKGL...     108   
3  A0A5F9ZHA1    0  MTTMLQKSDSNASFLRAARAGNLDKVVEYLKGGIDINTCNQNGLNA...    1849   
4      A4D126    0  MEAGPPGSARPAEPGPCLSGQRGADHTASASLQSVAGTEPGRHPQA...     451   

   n_go_raw  
0         0  
1         0  
2         0  
3         0  
4        18  

Missing sequence: 0
Zero GO annotations (raw): 8
